# RescueVision - xBD preprocessing -> single tar

In [ ]:
import subprocess, sys, os, json, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = Path("/tmp/RescueVision")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/AliOmarAbdelhady/RescueVision.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

def mark(stage, ok=True, extra=""):
    """Progress ledger in /kaggle/working: survives kills where the log does not."""
    rec = {"t": time.strftime("%H:%M:%S"), "stage": stage, "ok": bool(ok), "extra": str(extra)[:1500]}
    with open(WORK / "_progress.txt", "a") as f:
        f.write(json.dumps(rec) + "\n")
    (WORK / "_status.json").write_text(json.dumps(rec, indent=1))
    print("[mark]", rec, flush=True)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
mark("setup:start")
# pin albumentations: newer versions (preinstalled on the GPU image) have a
# stricter shape check that rejects our verified-consistent (HxWx3, HxW) pairs;
# --upgrade forces the downgrade past pip's "already satisfied"; all transforms
# ALSO use is_check_shapes=False as a version-proof second layer
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "albumentations==2.0.8"], check=False)
import albumentations as _A
mark("setup:albumentations", extra=_A.__version__)
print("albumentations", _A.__version__)

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# num_workers=0: DataLoader workers died on the Kaggle GPU image; pin_memory
# off too (useless with workers=0, and a hard-kill vector on some drivers)
NW = 0
PIN = False
print("torch", torch.__version__, "| device:", DEVICE)
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)
mark("setup:done", extra=str(DEVICE))

# per-epoch heartbeat: every trainer that logs an epoch also lands in the marks,
# so a silent kill still tells us how far training got and with what metrics
import rescuevision.training.logging_utils as _lu
_orig_log = _lu.CSVLogger.log
def _marked_log(self, metrics, epoch):
    try:
        mark(f"epoch{epoch}", extra=str({k: round(float(v), 4) for k, v in metrics.items()
                                         if isinstance(v, (int, float))})[:400])
    except Exception:
        pass
    return _orig_log(self, metrics, epoch)
_lu.CSVLogger.log = _marked_log
print("epoch heartbeat patch applied")

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from rescuevision.data.xbd_parser import parse_xbd_json
from rescuevision.data.xbd_preprocess import generate_masks_for_tile, generate_crops_for_tile
from rescuevision.data.split_utils import generate_event_aware_splits
from rescuevision.utils.io import find_image_pairs
from rescuevision.utils.image import load_image

print("kaggle inputs:", os.listdir("/kaggle/input"))

# locate the xBD train root (dir that contains images/ + labels/) robustly
RAW = None
for cand in Path("/kaggle/input").rglob("labels"):
    if cand.is_dir() and (cand.parent / "images").is_dir():
        RAW = cand.parent
        break
assert RAW is not None, "could not find a dir with images/ + labels/ under /kaggle/input"
print("raw xBD root:", RAW)
mark("preproc:raw-found", extra=str(RAW))

OUT = Path("/tmp/processed_xbd")     # built in /tmp; only the final tar lands in working
IMG_SIZE = 512          # stored tile size (training resolution)
CROP_MARGIN = 16        # px margin around building bboxes (native resolution)

for sub in ["images/pre", "images/post", "masks/building_binary", "masks/damage_multiclass",
            "masks/change_binary", "crops/building_pre", "crops/building_post", "metadata"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

pairs = find_image_pairs(RAW / "images")
print(f"found {len(pairs)} pre/post image pairs")
mark("preproc:pairs", extra=str(len(pairs)))

In [ ]:
tile_rows, building_rows = [], []
crop_counter = 0
processed = 0
written_bytes = 0

def _png(path, arr):
    global written_bytes
    Image.fromarray(arr).save(path)
    written_bytes += path.stat().st_size

for pre_path, post_path in tqdm(pairs, desc="processing tiles", miniters=25):
    tile_id = pre_path.stem.replace("_pre_disaster", "")
    try:
        pre_tile = parse_xbd_json(RAW / "labels" / f"{tile_id}_pre_disaster.json", pre_path)
        post_tile = parse_xbd_json(RAW / "labels" / f"{tile_id}_post_disaster.json", post_path)
        pre_image = load_image(pre_path)
        post_image = load_image(post_path)
    except Exception as e:
        print(f"skip {tile_id}: {e}")
        continue

    h, w = pre_image.shape[:2]
    masks = generate_masks_for_tile(pre_tile, post_tile, h, w)

    # images: bilinear to 512 (PNG ~3x smaller than npy) | masks: nearest (ids preserved)
    _png(OUT / "images/pre" / f"{tile_id}.png",
         np.array(Image.fromarray(pre_image).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)))
    _png(OUT / "images/post" / f"{tile_id}.png",
         np.array(Image.fromarray(post_image).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)))
    for mname, m in [("building_binary", masks["building_binary"]),
                     ("damage_multiclass", masks["damage_multiclass"]),
                     ("change_binary", masks["change_binary"])]:
        _png(OUT / "masks" / mname / f"{tile_id}.png",
             np.array(Image.fromarray(m).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)))

    # xBD filename carries the event: "{disaster-name}_{id}" -> derive so the
    # event-aware split always groups tiles correctly (pre-JSON metadata lacks it)
    disaster_name = tile_id.rsplit("_", 1)[0] if "_" in tile_id else tile_id
    try:
        import json as _json
        _meta = _json.loads((RAW / "labels" / f"{tile_id}_post_disaster.json").read_text()).get("metadata", {})
        disaster_type = _meta.get("disaster_type", "") or ""
    except Exception:
        disaster_type = ""

    tile_rows.append({
        "tile_id": tile_id,
        "disaster_name": disaster_name,
        "disaster_type": disaster_type,
        "num_buildings": pre_tile.num_buildings,
        "post_num_buildings": post_tile.num_buildings if post_tile else 0,
    })

    # per-building crops at native resolution (only usable damage labels)
    if post_tile is not None:
        for crop in generate_crops_for_tile(pre_tile, post_tile, pre_image, post_image,
                                            margin=CROP_MARGIN):
            if crop["damage_id"] is None or crop["damage_id"] == 255:
                continue  # un-classified buildings are filtered at train time anyway
            crop_id = f"crop_{crop_counter:07d}"
            crop_counter += 1
            _png(OUT / "crops/building_pre" / f"{crop_id}.png", crop["pre_crop"])
            _png(OUT / "crops/building_post" / f"{crop_id}.png", crop["post_crop"])
            building_rows.append({
                "crop_id": crop_id,
                "tile_id": tile_id,
                "label": crop["damage_label"],
                "label_id": crop["damage_id"],
                "area": crop["area"],
                "disaster_name": crop["disaster_name"],
                "disaster_type": crop["disaster_type"],
                "pre_crop_path": f"crops/building_pre/{crop_id}.png",
                "post_crop_path": f"crops/building_post/{crop_id}.png",
            })

    processed += 1
    if processed % 200 == 0:
        mark("preproc:progress", extra=f"{processed}/{len(pairs)} tiles, {written_bytes/1e9:.2f}GB")

print(f"processed {processed}/{len(pairs)} tiles | {len(building_rows)} classified building crops | {written_bytes/1e9:.2f} GB")
mark("preproc:tiles-done", extra=f"{processed} tiles, {len(building_rows)} crops, {written_bytes/1e9:.2f}GB")

In [ ]:
tiles_df = pd.DataFrame(tile_rows)
tiles_df.to_csv(OUT / "metadata/tiles.csv", index=False)

buildings_df = pd.DataFrame(building_rows)
buildings_df.to_csv(OUT / "metadata/buildings.csv", index=False)

# event-aware splits: all tiles of one disaster go to the same split (no leakage)
splits = generate_event_aware_splits(
    OUT / "metadata/tiles.csv", OUT / "metadata/splits_event_aware.csv")

print(splits["split"].value_counts())
print("\nbuildings per damage class:")
print(buildings_df["label"].value_counts())
mark("preproc:splits", extra=str(splits["split"].value_counts().to_dict()))

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# sanity overlay for the first processed tile -> straight to working (tiny)
tid = tiles_df.iloc[0]["tile_id"]
img = np.array(Image.open(OUT / "images/post" / f"{tid}.png"))
dmg = np.array(Image.open(OUT / "masks/damage_multiclass" / f"{tid}.png"))
bld = np.array(Image.open(OUT / "masks/building_binary" / f"{tid}.png"))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img); axes[0].set_title(f"post {tid}")
axes[1].imshow(bld, cmap="gray"); axes[1].set_title("building mask")
axes[2].imshow(dmg, vmin=0, vmax=4, cmap="turbo"); axes[2].set_title("damage mask (0-4)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.savefig(WORK / "sanity_tile.png", dpi=110, bbox_inches="tight")
plt.show()

from collections import Counter
mask_vals = Counter(dmg.flatten().tolist())
print("damage mask value counts (tile 0):", dict(mask_vals))

In [ ]:
# Pack the whole tree into ONE tar in working: kernel-source attach of a few files
# mounts reliably (the loose 170k-file tree NEVER mounted - that killed v1-v4).
import subprocess, json as _json
TAR_PATH = WORK / "processed_xbd.tar"
subprocess.run(["tar", "-cf", str(TAR_PATH), "-C", "/tmp", "processed_xbd"], check=True)
n_files = sum(1 for _ in OUT.rglob("*") if _.is_file())
summary = {
    "tiles": int(len(tiles_df)),
    "building_crops": int(len(buildings_df)),
    "tree_files": int(n_files),
    "tree_gb": round(written_bytes / 1e9, 2),
    "tar_gb": round(TAR_PATH.stat().st_size / 1e9, 2),
    "splits": splits["split"].value_counts().to_dict(),
}
(WORK / "preprocess_summary.json").write_text(_json.dumps(summary, indent=2))
print(_json.dumps(summary, indent=2))
print("working dir:", sorted(p.name for p in WORK.iterdir()))
mark("preproc:tar-done", extra=f"{summary['tar_gb']}GB")
print("NOTE: working dir must contain ONLY ~5 files (tar + sanity + summary + marks)")